# Часть 2. Получение и обработка данных по численности населения ИО

Мы загрузили таблицы в формате xls с сайта Росстата. Адрес: https://38.rosstat.gov.ru/folder/167937. 
Файлы "Численность населения по полу и возрасту на начало N года", где N от 2016 до 2024 включительно. Всего 9 файлов.
Переименовали для удобства в:

- pol_voz_2016
- pol_voz_2017
- pol_voz_2018
- pol_voz_2019
- pol_voz_2020
- pol_voz_2021
- pol_voz_2022
- pol_voz_2023
- pol_voz_2024

Cобрали в один файл pol_voz_2016_2024

In [1]:
import pandas as pd
from pathlib import Path
import base64
from IPython.display import HTML, display

In [2]:
# Находим и читаем файл
file = list(Path(".").rglob("pol_voz_2016_2024.xlsx"))[0]
sheets = pd.read_excel(file, sheet_name=None)

# Создаем переменные для каждого листа
for name, df in sheets.items():
    globals()[f"df_{name}"] = df
    print(f"✓ df_{name} - {df.shape}")

✓ df_2016 - (126, 11)
✓ df_2017 - (126, 11)
✓ df_2018 - (126, 11)
✓ df_2019 - (126, 11)
✓ df_2020 - (126, 11)
✓ df_2021 - (126, 11)
✓ df_2022 - (126, 13)
✓ df_2023 - (126, 11)
✓ df_2024 - (126, 11)


In [3]:
df_2018.head()

,Численность населения по полу и возрасту на 1 января 2018 года,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,Муниципальные образования Иркутской области,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Возраст (лет),№№ строки,Все население,NaN,NaN,Городское население,NaN,NaN,Сельское население,NaN,NaN
2,NaN,NaN,мужчины и женщины,мужчины,женщины,мужчины и женщины,мужчины,женщины,мужчины и женщины,мужчины,женщины
3,А,Б,1,2,3,4,5,6,7,8,9
4,Все население,1,2404195,1111049,1293146,1894053,860701,1033352,510142,250348,259794


### Обработка и очистка

In [4]:

def process_population_data_simple(df, year):
    """
    Обработка таблицы населения Росстата
    
    Шаги:
    1. Оставляем первые 11 столбцов (возраст + 10 колонок с населением)
    2. Заполняем пустые ячейки во 2-й и 3-й строках (объединенные ячейки в шапке)
    3. Объединяем 2-ю и 3-ю строки в названия столбцов
    4. Удаляем первые 4 строки (шапку), оставляя только данные
    5. Добавляем колонку с годом
    6. Преобразуем числа в числовой формат (кроме колонки с возрастом)
    7. Удаляем строки, где возраст в формате "число-число" (0-4, 90-94 и т.д.)
    8. Удаляем 2-й столбец
    """
    
    # 1. Оставляем только первые 11 столбцов
    df = df.iloc[:, :11]
    
    # 2. Заполняем пустоты во 2 и 3 строках (индексы 1 и 2)
    df.iloc[1:3] = df.iloc[1:3].ffill(axis=1)
    
    # 3. Формируем названия столбцов из 2-й и 3-й строк
    headers = []
    for col in range(11):
        main_header = str(df.iloc[2, col]) if pd.notna(df.iloc[2, col]) else ''
        sub_header = str(df.iloc[1, col]) if pd.notna(df.iloc[1, col]) else ''
        
        if sub_header and sub_header != main_header:
            headers.append(f"{main_header}_{sub_header}" if main_header else sub_header)
        else:
            headers.append(main_header if main_header else f'col_{col}')
    
    # 4. Удаляем первые 4 строки (индексы 0,1,2,3)
    df = df.iloc[4:].reset_index(drop=True)
    
    # 5. Присваиваем новые названия столбцам
    df.columns = headers
    
    # 6. Добавляем колонку с годом
    df['Год'] = year
    
    # 7. Преобразуем числа в числовой формат (колонку возраста не трогаем)
    for col in df.columns:
        if col != 'Год' and col != df.columns[0]:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 8. Удаляем строки с возрастными интервалами (содержат дефис)
    age_col = df.columns[0]
    df = df[~df[age_col].astype(str).str.contains('-', na=False)]
    
    # 9. Удаляем 2-й столбец (индекс 1)
    df = df.drop(df.columns[1], axis=1)
    
    return df

# Обрабатываем каждый год с 2016 по 2024
for year in range(2016, 2025):
    if f"df_{year}" in globals():
        globals()[f"df_{year}_clean"] = process_population_data_simple(globals()[f"df_{year}"], year)
        print(f"✓ df_{year}_clean")


✓ df_2016_clean
✓ df_2017_clean
✓ df_2018_clean
✓ df_2019_clean
✓ df_2020_clean
✓ df_2021_clean
✓ df_2022_clean
✓ df_2023_clean
✓ df_2024_clean


In [5]:
df_2022_clean.head()

,Возраст (лет),мужчины и женщины_Все население,мужчины_Все население,женщины_Все население,мужчины и женщины_Городское население,мужчины_Городское население,женщины_Городское население,мужчины и женщины_Сельское население,мужчины_Сельское население,женщины_Сельское население,Год
0,Все население,2363447,1086115,1277332,1834514,827372,1007142,528933,258743,270190,2022
1,0,25265,13011,12254,18799,9724,9075,6466,3287,3179,2022
2,1,25900,13262,12638,19555,9983,9572,6345,3279,3066,2022
3,2,27230,13987,13243,19750,10094,9656,7480,3893,3587,2022
4,3,29563,15329,14234,22280,11492,10788,7283,3837,3446,2022


### Фильтруем

In [6]:
def filter_columns(df, year):
    """
    Фильтрация столбцов: оставляем первые 4 и последний (год)
    Переименовываем столбцы
    """
    # Оставляем первые 4 столбца
    df_filtered = df.iloc[:, :4].copy()
    
    # Переименовываем
    df_filtered.columns = ['возраст', 'мужчины_и_женщины', 'мужчины', 'женщины']
    
    # Добавляем столбец с годом
    df_filtered['год'] = year
    
    # Находим строку с "все население" и перемещаем в конец
    mask = df_filtered['возраст'].astype(str).str.contains('все население|Все население|ВСЕ НАСЕЛЕНИЕ', na=False)
    row_all = df_filtered[mask]
    df_filtered = df_filtered[~mask]
    df_filtered = pd.concat([df_filtered, row_all], ignore_index=True)
    
    return df_filtered

# Применяем ко всем годам
for year in range(2016, 2025):
    df_name = f"df_{year}_clean"
    if df_name in globals():
        globals()[f"df_{year}_all"] = filter_columns(globals()[df_name], year)
        print(f"✓ df_{year}_all")

✓ df_2016_all
✓ df_2017_all
✓ df_2018_all
✓ df_2019_all
✓ df_2020_all
✓ df_2021_all
✓ df_2022_all
✓ df_2023_all
✓ df_2024_all


In [7]:
df_2024_all.head()

,возраст,мужчины_и_женщины,мужчины,женщины,год
0,0,23160,11870,11290,2024
1,1,24376,12583,11793,2024
2,2,25897,13331,12566,2024
3,3,26552,13630,12922,2024
4,4,27786,14311,13475,2024


### Расчет численности по категориям возрастов

Проверим верно ли рассичтана общая численность как сумма численностей всех возрастов

In [8]:
def calculate_sum(df):
    """
    Рассчитывает сумму для возраста 0-100 и добавляет строку 'сумма'
    """
    # Создаем копию
    df_result = df.copy()
    
    # Суммируем строки с индексами от 0 до 100 (включительно)
    sum_men_women = df_result.loc[0:100, 'мужчины_и_женщины'].sum()
    sum_men = df_result.loc[0:100, 'мужчины'].sum()
    sum_women = df_result.loc[0:100, 'женщины'].sum()
    
    # Создаем строку с суммой
    sum_row = pd.DataFrame({
        'возраст': ['сумма'],
        'мужчины_и_женщины': [sum_men_women],
        'мужчины': [sum_men],
        'женщины': [sum_women],
        'год': [df_result['год'].iloc[0]]
    })
    
    # Добавляем строку в конец
    df_result = pd.concat([df_result, sum_row], ignore_index=True)
    
    return df_result

# Применяем ко всем годам
for year in range(2016, 2025):
    df_name = f"df_{year}_all"
    if df_name in globals():
        globals()[f"df_{year}_sum"] = calculate_sum(globals()[df_name])
        print(f"✓ df_{year}_sum")

# Выводим последние 2 строки каждого датасета
for year in range(2016, 2025):
    df_name = f"df_{year}_sum"
    if df_name in globals():
        print(f"\n{df_name}:")
        print(globals()[df_name].tail(2))

✓ df_2016_sum
✓ df_2017_sum
✓ df_2018_sum
✓ df_2019_sum
✓ df_2020_sum
✓ df_2021_sum
✓ df_2022_sum
✓ df_2023_sum
✓ df_2024_sum

df_2016_sum:
           возраст  мужчины_и_женщины  мужчины  женщины   год
101  Все население            2412800  1115508  1297292  2016
102          сумма            2412800  1115508  1297292  2016

df_2017_sum:
           возраст  мужчины_и_женщины  мужчины  женщины   год
101  Все население            2408901  1113729  1295172  2017
102          сумма            2408901  1113729  1295172  2017

df_2018_sum:
           возраст  мужчины_и_женщины  мужчины  женщины   год
101  Все население            2404195  1111049  1293146  2018
102          сумма            2404195  1111049  1293146  2018

df_2019_sum:
           возраст  мужчины_и_женщины  мужчины  женщины   год
101  Все население            2397763  1107831  1289932  2019
102          сумма            2397763  1107831  1289932  2019

df_2020_sum:
           возраст  мужчины_и_женщины  мужчины  женщины   го

Расчетные и табличные суммы общей численности совпадают!

In [9]:
# Определяем возрастные категории (диапазоны лет)
age_categories = {
    '0-4_лет': list(range(0, 5)),
    '5-6_лет': list(range(5, 7)),
    '7-14_лет': list(range(7, 15)),
    '15-17_лет': list(range(15, 18)),
    '18-24_лет': list(range(18, 25)),
    '25-34_лет': list(range(25, 35)),
    '35-44_лет': list(range(35, 45)),
    '45-54_лет': list(range(45, 55)),
    '55-64_лет': list(range(55, 65)),
    '65+': list(range(65, 101))
}

results = []

for year in range(2016, 2025):
    df_name = f"df_{year}_sum"
    if df_name in globals():
        df = globals()[df_name]
        row = {'Год': year}
        
        # Общие итоги
        row['Всего'] = df.loc[0:100, 'мужчины_и_женщины'].sum()
        row['м_Всего'] = df.loc[0:100, 'мужчины'].sum()
        row['ж_Всего'] = df.loc[0:100, 'женщины'].sum()
        
        # Для каждой возрастной категории
        for cat_name, ages in age_categories.items():
            row[cat_name] = df.loc[ages, 'мужчины_и_женщины'].sum()
            row[f'м_{cat_name}'] = df.loc[ages, 'мужчины'].sum()
            row[f'ж_{cat_name}'] = df.loc[ages, 'женщины'].sum()
        
        # Категория 0-14 лет
        ages_0_14 = list(range(0, 15))
        row['0-14_лет'] = df.loc[ages_0_14, 'мужчины_и_женщины'].sum()
        row['м_0-14_лет'] = df.loc[ages_0_14, 'мужчины'].sum()
        row['ж_0-14_лет'] = df.loc[ages_0_14, 'женщины'].sum()
        
        # Категория 0-17 лет
        ages_0_17 = list(range(0, 18))
        row['0-17_лет'] = df.loc[ages_0_17, 'мужчины_и_женщины'].sum()
        row['м_0-17_лет'] = df.loc[ages_0_17, 'мужчины'].sum()
        row['ж_0-17_лет'] = df.loc[ages_0_17, 'женщины'].sum()
        
        # Категория 15+
        ages_15_plus = list(range(15, 101))
        row['15+'] = df.loc[ages_15_plus, 'мужчины_и_женщины'].sum()
        row['м_15+'] = df.loc[ages_15_plus, 'мужчины'].sum()
        row['ж_15+'] = df.loc[ages_15_plus, 'женщины'].sum()
        
        # Категория 18+
        ages_18_plus = list(range(18, 101))
        row['18+'] = df.loc[ages_18_plus, 'мужчины_и_женщины'].sum()
        row['м_18+'] = df.loc[ages_18_plus, 'мужчины'].sum()
        row['ж_18+'] = df.loc[ages_18_plus, 'женщины'].sum()
        
        results.append(row)

# Создаём DataFrame
df_age_groups = pd.DataFrame(results)

# Порядок колонок
base_cols = ['Год', 'Всего', 'м_Всего', 'ж_Всего']
age_cols = []
for cat in age_categories.keys():
    age_cols.extend([cat, f'м_{cat}', f'ж_{cat}'])
extra_cols = ['0-14_лет', 'м_0-14_лет', 'ж_0-14_лет', 
              '0-17_лет', 'м_0-17_лет', 'ж_0-17_лет',
              '15+', 'м_15+', 'ж_15+', 
              '18+', 'м_18+', 'ж_18+']
column_order = base_cols + age_cols + extra_cols
column_order = [c for c in column_order if c in df_age_groups.columns]
df_age_groups = df_age_groups[column_order]

print("ИТОГОВЫЙ ДАТАСЕТ С РАЗБИВКОЙ ПО ПОЛУ И ВОЗРАСТУ:")
print(df_age_groups)

# Проверка
print("\n" + "="*60)
print("ПРОВЕРКА (2024 год):")
print("="*60)
row_2024 = df_age_groups[df_age_groups['Год'] == 2024].iloc[0]
print(f"Всего: {row_2024['Всего']:,.0f}")
print(f"0-17 лет: {row_2024['0-17_лет']:,.0f}")
print(f"18+ лет: {row_2024['18+']:,.0f}")
print(f"0-17 + 18+ = {row_2024['0-17_лет'] + row_2024['18+']:,.0f}")
print(f"м_Всего: {row_2024['м_Всего']:,.0f}")
print(f"ж_Всего: {row_2024['ж_Всего']:,.0f}")

ИТОГОВЫЙ ДАТАСЕТ С РАЗБИВКОЙ ПО ПОЛУ И ВОЗРАСТУ:
    Год    Всего  м_Всего  ж_Всего  0-4_лет  м_0-4_лет  ж_0-4_лет  5-6_лет  \
0  2016  2412800  1115508  1297292   184508      94508      90000    68365   
1  2017  2408901  1113729  1295172   183080      94230      88850    70049   
2  2018  2404195  1111049  1293146   177110      91120      85990    73839   
3  2019  2397763  1107831  1289932   170405      87775      82630    74605   
4  2020  2391193  1106100  1285093   161919      83410      78509    73060   
5  2021  2375021  1098190  1276831   152059      78358      73701    72108   
6  2022  2363447  1086115  1277332   138740      71294      67446    69334   
7  2023  2344360  1075988  1268372   135179      69686      65493    66444   
8  2024  2330537  1068992  1261545   127771      65725      62046    61716   

   м_5-6_лет  ж_5-6_лет  ...  ж_0-14_лет  0-17_лет  м_0-17_лет  ж_0-17_лет  \
0      34959      33406  ...      236115    558902      286518      272384   
1      35690  

### Вычисляем среднегодовые численности

In [10]:
# РАСЧЁТ СРЕДНЕГОДОВЫХ ЗНАЧЕНИЙ
# ============================================================================

# Создаем список для хранения среднегодовых значений
mean_results = []

# Для каждого года с 2016 по 2023
for year in range(2016, 2024):
    current = df_age_groups[df_age_groups['Год'] == year].iloc[0]
    next_year = df_age_groups[df_age_groups['Год'] == year + 1].iloc[0]
    
    mean_row = {'Год': year}
    
    for col in df_age_groups.columns:
        if col != 'Год':
            mean_row[col] = int((current[col] + next_year[col]) / 2 + 0.5)
    
    mean_results.append(mean_row)

# Добавляем 2024 год как есть
mean_results.append(df_age_groups[df_age_groups['Год'] == 2024].iloc[0].to_dict())

# Создаем датасет
irkutsk_population_avg = pd.DataFrame(mean_results)

print("\nСРЕДНЕГОДОВЫЕ ПОКАЗАТЕЛИ ЧИСЛЕННОСТИ НАСЕЛЕНИЯ:")
print(irkutsk_population_avg)

# Проверка для 2016 года
print("\n" + "="*60)
print("ПРОВЕРКА (2016 год):")
print("="*60)
print(f"2016 исходный: {df_age_groups.loc[df_age_groups['Год']==2016, 'Всего'].values[0]:,.0f}")
print(f"2017 исходный: {df_age_groups.loc[df_age_groups['Год']==2017, 'Всего'].values[0]:,.0f}")
print(f"2016 среднегодовой: {irkutsk_population_avg.loc[irkutsk_population_avg['Год']==2016, 'Всего'].values[0]:,.0f}")


СРЕДНЕГОДОВЫЕ ПОКАЗАТЕЛИ ЧИСЛЕННОСТИ НАСЕЛЕНИЯ:
    Год    Всего  м_Всего  ж_Всего  0-4_лет  м_0-4_лет  ж_0-4_лет  5-6_лет  \
0  2016  2410851  1114619  1296232   183794      94369      89425    69207   
1  2017  2406548  1112389  1294159   180095      92675      87420    71944   
2  2018  2400979  1109440  1291539   173758      89448      84310    74222   
3  2019  2394478  1106966  1287513   166162      85593      80570    73833   
4  2020  2383107  1102145  1280962   156989      80884      76105    72584   
5  2021  2369234  1092153  1277082   145400      74826      70574    70721   
6  2022  2353904  1081052  1272852   136960      70490      66470    67889   
7  2023  2337449  1072490  1264959   131475      67706      63770    64080   
8  2024  2330537  1068992  1261545   127771      65725      62046    61716   

   м_5-6_лет  ж_5-6_лет  ...  ж_0-14_лет  0-17_лет  м_0-17_лет  ж_0-17_лет  \
0      35325      33883  ...      237910    563166      288921      274246   
1      36680  

### Сохраняем irkutsk_population_2016_2024

In [11]:
irkutsk_population_avg.columns

Index(['Год', 'Всего', 'м_Всего', 'ж_Всего', '0-4_лет', 'м_0-4_лет',
       'ж_0-4_лет', '5-6_лет', 'м_5-6_лет', 'ж_5-6_лет', '7-14_лет',
       'м_7-14_лет', 'ж_7-14_лет', '15-17_лет', 'м_15-17_лет', 'ж_15-17_лет',
       '18-24_лет', 'м_18-24_лет', 'ж_18-24_лет', '25-34_лет', 'м_25-34_лет',
       'ж_25-34_лет', '35-44_лет', 'м_35-44_лет', 'ж_35-44_лет', '45-54_лет',
       'м_45-54_лет', 'ж_45-54_лет', '55-64_лет', 'м_55-64_лет', 'ж_55-64_лет',
       '65+', 'м_65+', 'ж_65+', '0-14_лет', 'м_0-14_лет', 'ж_0-14_лет',
       '0-17_лет', 'м_0-17_лет', 'ж_0-17_лет', '15+', 'м_15+', 'ж_15+', '18+',
       'м_18+', 'ж_18+'],
      dtype='object')

In [12]:

# 1. Создаем пустой список для новых строк
rows = []

# 2. Для каждого года создаем три строки (всего, мужчины, женщины)
for year in irkutsk_population_avg['Год'].unique():
    row_total = {'Год': year, 'Пол': 'всего'}
    row_male = {'Год': year, 'Пол': 'мужчины'}
    row_female = {'Год': year, 'Пол': 'женщины'}
    
    for col in irkutsk_population_avg.columns:
        if col == 'Год':
            continue
            
        # Упрощаем название возрастной группы
        if col == 'Всего':
            age_group = 'Всего'
            # Значение для 'всего'
            row_total[age_group] = irkutsk_population_avg.loc[irkutsk_population_avg['Год'] == year, col].values[0]
        elif col == 'м_Всего':
            age_group = 'Всего'
            row_male[age_group] = irkutsk_population_avg.loc[irkutsk_population_avg['Год'] == year, col].values[0]
        elif col == 'ж_Всего':
            age_group = 'Всего'
            row_female[age_group] = irkutsk_population_avg.loc[irkutsk_population_avg['Год'] == year, col].values[0]
        elif col.startswith('м_'):
            age_group = col[2:].replace('_лет', '')
            row_male[age_group] = irkutsk_population_avg.loc[irkutsk_population_avg['Год'] == year, col].values[0]
        elif col.startswith('ж_'):
            age_group = col[2:].replace('_лет', '')
            row_female[age_group] = irkutsk_population_avg.loc[irkutsk_population_avg['Год'] == year, col].values[0]
        else:
            age_group = col.replace('_лет', '')
            row_total[age_group] = irkutsk_population_avg.loc[irkutsk_population_avg['Год'] == year, col].values[0]
    
    rows.append(row_total)
    rows.append(row_male)
    rows.append(row_female)

# 3. Создаем новый DataFrame
df_result = pd.DataFrame(rows)

# 4. Упорядочиваем колонки: Год, Пол, затем возрастные группы (можно настроить порядок)
age_columns = [c for c in df_result.columns if c not in ['Год', 'Пол']]
irkutsk_population_2016_2024 = df_result[['Год', 'Пол'] + age_columns]

# Результат: df_result

In [13]:
irkutsk_population_2016_2024.head()

,Год,Пол,Всего,0-4,5-6,7-14,15-17,18-24,25-34,35-44,45-54,55-64,65+,0-14,0-17,15+,18+
0,2016,всего,2410851,183794,69207,235186,74979,190183,409323,348260,291441,318713,289765,488187,563166,1922664,1847685
1,2016,мужчины,1114619,94369,35325,120584,38644,94937,206341,166344,134731,133076,90271,250277,288921,864342,825698
2,2016,женщины,1296232,89425,33883,114603,36336,95247,202982,181917,156710,185638,199495,237910,274246,1058322,1021987
3,2017,всего,2406548,180095,71944,242241,76615,179732,400958,351781,286420,317414,299350,494280,570895,1912269,1835654
4,2017,мужчины,1112389,92675,36680,124173,39412,89836,202508,168114,132457,132864,93673,253527,292939,858862,819451


In [14]:
# Функция для создания ссылки на скачивание
def create_download_link(df, filename):
    csv = df.to_csv(index=False, encoding='utf-8-sig')
    b64 = base64.b64encode(csv.encode()).decode()
    display(HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}">📥 Скачать {filename}</a>'))

# Сохраняем irkutsk_population_2016_2024 через base64 ссылку
print("Скачать irkutsk_population_2016_2024:")
create_download_link(irkutsk_population_2016_2024, 'irkutsk_population_2016_2024.csv')

# Сохраняем irkutsk_population_avg через base64 ссылку
print("\nСкачать irkutsk_population_avg:")
create_download_link(irkutsk_population_avg, 'irkutsk_population_avg.csv')

# Если нужен df_age_groups
# print("\nСкачать df_age_groups:")
# create_download_link(df_age_groups, 'df_age_groups.csv')

Скачать irkutsk_population_2016_2024:



Скачать irkutsk_population_avg:


In [15]:
irkutsk_population_2016_2024.columns

Index(['Год', 'Пол', 'Всего', '0-4', '5-6', '7-14', '15-17', '18-24', '25-34',
       '35-44', '45-54', '55-64', '65+', '0-14', '0-17', '15+', '18+'],
      dtype='object')

In [16]:
# Простая проверка согласованности данных
print("="*50)
print("ПРОВЕРКА СОГЛАСОВАННОСТИ ДАННЫХ")
print("="*50)

# Список возрастных групп
age_groups = ['0-4', '5-6', '7-14', '15-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']

# Проверяем для каждого пола отдельно
for sex in ['всего', 'мужчины', 'женщины']:
    df_sex = irkutsk_population_2016_2024[irkutsk_population_2016_2024['Пол'] == sex]
    
    print(f"\n--- {sex.upper()} ---")
    
    # Проверка: Всего = сумма возрастных групп
    sum_ages = df_sex[age_groups].sum(axis=1)
    max_diff = (df_sex['Всего'] - sum_ages).abs().max()
    mean_diff = (df_sex['Всего'] - sum_ages).abs().mean()
    
    print(f"Всего = сумма возрастов:")
    print(f"  Макс. расхождение: {max_diff:.2f}")
    print(f"  Ср. расхождение: {mean_diff:.2f}")
    
    # Проверка: 0-14 = 0-4 + 5-6 + 7-14
    if '0-14' in df_sex.columns:
        max_diff_014 = (df_sex['0-14'] - (df_sex['0-4'] + df_sex['5-6'] + df_sex['7-14'])).abs().max()
        print(f"0-14 = 0-4+5-6+7-14: макс. расхождение {max_diff_014:.2f}")

# Общая оценка
print("\n" + "="*50)
print("ИТОГ:")
if max_diff < 0.1 and max_diff_014 < 0.1:
    print("✅ Данные полностью согласованы!")
else:
    print("⚠️ В данных есть расхождения (возможно из-за округлений)")

ПРОВЕРКА СОГЛАСОВАННОСТИ ДАННЫХ

--- ВСЕГО ---
Всего = сумма возрастов:
  Макс. расхождение: 3.00
  Ср. расхождение: 1.67
0-14 = 0-4+5-6+7-14: макс. расхождение 1.00

--- МУЖЧИНЫ ---
Всего = сумма возрастов:
  Макс. расхождение: 4.00
  Ср. расхождение: 2.44
0-14 = 0-4+5-6+7-14: макс. расхождение 1.00

--- ЖЕНЩИНЫ ---
Всего = сумма возрастов:
  Макс. расхождение: 4.00
  Ср. расхождение: 2.56
0-14 = 0-4+5-6+7-14: макс. расхождение 1.00

ИТОГ:
⚠️ В данных есть расхождения (возможно из-за округлений)
